# 🔤 27 · Attention, Transformers, and Diffusion
### From attending to the most relevant parts of an image to generating new ones, one denoising step at a time

**Part I · Quantifying and modern architectures** · [⟵ 26 Colocalization & spatial statistics](26_colocalization_and_spatial_statistics.ipynb) · [🏠 Course home](../README.md) · [28 Foundation models for microscopy ⟶](28_foundation_models_for_microscopy.ipynb)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/27_attention_transformers_diffusion.ipynb)

| | |
|---|---|
| **Hardware target** | CPU; every training run in this notebook finishes in well under a minute |
| **Core frameworks** | NumPy, PyTorch (CPU-only), Matplotlib |
| **Prerequisites** | Module 16 (CNNs & deep learning basics), Module 17 (latent spaces, autoencoders), Module 21 (VAEs and GANs — the other generative paradigms) |
| **Estimated time** | 90-110 minutes |
| **02-740 topics covered** | self-attention and the Vision Transformer (ViT) architecture built from scratch (patch embedding, multi-head self-attention, a CLS-token classifier); diffusion models built from scratch (the forward noising process, the learned reverse/denoising process, a DDPM-style toy generator); why these two architectures underpin much of modern bioimage analysis |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> All images, point clouds and training data in this notebook are SIMULATED by code — nothing is downloaded.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- explain the core idea of self-attention — a content-based, learned lookup over every other element in a sequence — and implement scaled dot-product attention and multi-head self-attention from scratch;
- explain how a Vision Transformer (ViT) applies this idea to images via patch embedding, a learnable classification token, and positional information, and train a minimal ViT-style classifier on a task that genuinely requires combining evidence from across the whole image;
- explain the forward (noising) and reverse (learned denoising) processes behind diffusion models, and implement and train a small DDPM-style diffusion model from scratch;
- compare self-attention/transformers and diffusion models against the convolutional (Module 16) and VAE/GAN (Module 21) approaches already seen in the course, stating what each new idea buys you and at what cost;
- explain, at a conceptual level, why attention-based backbones and diffusion-style training underpin many of the generalist foundation models now reshaping bioimage analysis (Module 28).

## 🗺️ Contents

1. From convolution to attention: a different way of combining information
2. Self-attention from scratch: queries, keys, and values
3. From pixels to patches: a minimal Vision Transformer (ViT)
4. Generative modelling by denoising: the diffusion process
5. A DDPM-style toy diffusion model, trained from scratch
6. Where this is going: attention and diffusion in modern bioimage analysis

## 🧫 The biological question

A convolutional filter (Module 16) only ever looks at a small, fixed neighbourhood around each pixel — excellent for local texture, but with no built-in way to relate a cell on one side of the field of view to a structurally similar one far across it, except by stacking many layers until the receptive field eventually grows large enough. Could a network instead learn, directly, *which* other locations in an image are worth paying attention to when interpreting any given one? And on the generative side — Module 21 built models (VAEs, GANs) that generate a new image in one shot from a latent code — what if, instead, a model generated an image gradually, starting from pure noise and undoing a little of it at a time? Both questions turn out to have reshaped bioimage analysis over the last few years, under the names **self-attention** and **diffusion models**, and this notebook builds minimal, working versions of both from scratch.

---
## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
try:
    import torch
    print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
except ImportError:
    print('PyTorch not installed yet — the next cell installs it if needed.')


In [ ]:
!pip install -q scikit-image torch
# ── Dependencies ── (Colab already has most of these; we install only what is missing)


In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
import torch
import torch.nn as nn
SEED = 19
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11


### 🧰 Course toolkit

We reuse `simulate_nuclei_image` and `show_image_grid`; everything specific to attention, transformers and diffusion in this notebook is built from scratch below, since it is new material for the course.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment

SHAPE_CLASSES = ("round", "elongated", "kidney", "ring", "dividing", "irregular")


def _draw_blob(canvas: np.ndarray, cy: float, cx: float, r: float, shape: str,
                rng: np.random.Generator, angle: float = 0.0) -> None:
    """Add one synthetic nucleus/cell of the given shape class onto ``canvas`` (in place, additive)."""
    size = canvas.shape[0]
    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
    y, x = yy - cy, xx - cx
    ca, sa = np.cos(angle), np.sin(angle)
    yr, xr = ca * y + sa * x, -sa * y + ca * x  # rotate into the blob's own frame
    if shape == "round":
        d = (yr / r) ** 2 + (xr / r) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "elongated":
        d = (yr / (r * 0.55)) ** 2 + (xr / (r * 1.7)) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "kidney":
        d = (yr / r) ** 2 + (xr / r) ** 2
        dent = np.exp(-((yr - 0.0) ** 2 + (xr - r * 0.9) ** 2) / (2 * (r * 0.5) ** 2))
        canvas += np.clip(np.exp(-d * 2.0) - 0.9 * dent, 0, None)
    elif shape == "ring":
        d = np.sqrt(yr ** 2 + xr ** 2)
        canvas += np.exp(-((d - r) ** 2) / (2 * (r * 0.28) ** 2))
    elif shape == "dividing":
        sep = r * 1.1
        for sign in (-1, 1):
            d = ((yr - sign * sep * 0.5) / (r * 0.75)) ** 2 + (xr / (r * 0.75)) ** 2
            canvas += np.exp(-d * 2.0)
    else:  # irregular: a round blob perturbed by a few random bumps
        d = (yr / r) ** 2 + (xr / r) ** 2
        base = np.exp(-d * 2.0)
        for _ in range(3):
            bx, by = rng.uniform(-r, r, size=2)
            bump = np.exp(-(((yr - by) ** 2 + (xr - bx) ** 2) / (2 * (r * 0.45) ** 2)))
            base = np.clip(base + 0.5 * bump, 0, 1.3)
        canvas += base


def add_gaussian_noise(image: np.ndarray, sigma: float, rng: np.random.Generator) -> np.ndarray:
    """Additive Gaussian (read) noise: the classic noise floor of a camera sensor."""
    return image + rng.normal(0.0, sigma, size=image.shape)


def blur_image(image: np.ndarray, psf: np.ndarray) -> np.ndarray:
    """Convolve an image with a point-spread function (microscope optical blur)."""
    return ndi.convolve(image, psf, mode="reflect")


def gaussian_psf(sigma: float, radius: int | None = None) -> np.ndarray:
    """A normalised 2-D Gaussian point-spread function (the microscope's blur kernel)."""
    if radius is None:
        radius = max(1, int(np.ceil(3 * sigma)))
    ax = np.arange(-radius, radius + 1)
    xx, yy = np.meshgrid(ax, ax)
    k = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    return k / k.sum()


def simulate_nuclei_image(n_cells: int = 25, size: int = 160, touching: bool = False,
                           shape_classes: tuple[str, ...] | None = None,
                           psf_sigma: float = 1.2, background: float = 0.05,
                           noise_sigma: float = 0.03, seed: int = 0):
    """A SIMULATED single-channel fluorescence image of cell nuclei/cells.

    Returns ``image`` (size x size, background + PSF-blurred blobs + Gaussian noise),
    ``instance_mask`` (0 = background, 1..n_cells = one integer id per cell, the ground truth
    for segmentation/detection/tracking), ``centroids`` (n_cells x 2, row, col), ``shape_labels``
    (one of SHAPE_CLASSES per cell, the ground truth for classification), and ``boxes`` (n_cells x 4,
    ymin, xmin, ymax, xmax — the ground truth for detection).
    """
    rng = np.random.default_rng(seed)
    classes = shape_classes or SHAPE_CLASSES
    image = np.full((size, size), background, dtype=float)
    instance_mask = np.zeros((size, size), dtype=int)
    centroids, labels, boxes = [], [], []
    margin = size * 0.12
    min_sep = size * 0.11 if not touching else size * 0.065
    placed = []
    attempts = 0
    while len(placed) < n_cells and attempts < n_cells * 60:
        attempts += 1
        cy, cx = rng.uniform(margin, size - margin, size=2)
        if all(np.hypot(cy - py, cx - px) > min_sep for py, px in placed):
            placed.append((cy, cx))
    for i, (cy, cx) in enumerate(placed):
        shape = classes[rng.integers(len(classes))]
        r = rng.uniform(size * 0.028, size * 0.045)
        angle = rng.uniform(0, 2 * np.pi)
        single = np.zeros_like(image)
        _draw_blob(single, cy, cx, r, shape, rng, angle)
        image += single
        this_mask = single > 0.25
        instance_mask[this_mask & (instance_mask == 0)] = i + 1
        centroids.append((cy, cx))
        labels.append(shape)
        ys, xs = np.where(this_mask)
        if len(ys):
            boxes.append((ys.min(), xs.min(), ys.max(), xs.max()))
        else:
            boxes.append((cy - r, cx - r, cy + r, cx + r))
    image = blur_image(image, gaussian_psf(psf_sigma))
    image = add_gaussian_noise(image, noise_sigma, rng)
    image = np.clip(image, 0, None)
    return dict(image=image, instance_mask=instance_mask, centroids=np.array(centroids),
                shape_labels=np.array(labels), boxes=np.array(boxes), source="SIMULATED")


def show_image(ax, image: np.ndarray, title: str = "", cmap: str = "gray"):
    """Show one image on a given Matplotlib axis, titled, with ticks removed."""
    ax.imshow(image, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])


def show_image_grid(images, titles=None, ncols: int = 4, cmap: str = "gray", figsize_scale: float = 2.6):
    """A grid of images with Matplotlib, one subplot each, titles optional."""
    n = len(images)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_scale * ncols, figsize_scale * nrows))
    axes = np.atleast_1d(axes).ravel()
    for i, img in enumerate(images):
        show_image(axes[i], img, titles[i] if titles else "", cmap=cmap)
    for ax in axes[n:]:
        ax.axis("off")
    fig.tight_layout()
    return fig, axes

---

## 1 · From convolution to attention: a different way of combining information

A convolutional layer combines information using a *fixed, local, position-independent* rule: the same small kernel slides over every location, blind to anything outside its receptive field. **Self-attention** combines information using a *learned, global, content-dependent* rule instead: every element of a sequence (for images, every image patch) can directly look at every other element, and how much attention it pays to each one is computed from their actual content, not from a fixed geometric rule. This is a genuinely different inductive bias — slower to pick up simple local patterns than a convolutional filter tuned for exactly that, but able to relate distant parts of an input in a single layer, which is precisely what motivates the next three sections.

## 2 · Self-attention from scratch: queries, keys, and values

Given a sequence of $N$ vectors (image patches, in Section 3), self-attention computes three learned projections of each one — a **query** $Q$, a **key** $K$, and a **value** $V$ — and then, for every element, asks "how well does my query match every other element's key?" via a scaled dot product, turns those match scores into weights with a softmax, and returns a weighted sum of every element's value:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

The $\sqrt{d_k}$ scaling (dividing by the square root of the key dimension) keeps the dot products, and hence the softmax, from growing too large as the dimensionality $d_k$ increases — without it, large dot products push the softmax toward a one-hot output and the gradients it provides shrink toward zero.

In [ ]:
# ── 2.1 Scaled dot-product attention, implemented directly from the formula ──
def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = (Q @ K.transpose(-2, -1)) / np.sqrt(d_k)
    weights = torch.softmax(scores, dim=-1)
    return weights @ V, weights

✏️ **Your turn.** Remove the `/ np.sqrt(d_k)` scaling entirely (just use `scores = Q @ K.transpose(-2, -1)`) and re-run the toy example below with `token_dim` increased from 8 to 64. Do the attention weights become more sharply peaked (closer to one-hot) than with scaling restored? This is exactly the effect the $\sqrt{d_k}$ term is designed to counteract.

In [ ]:
# ── 2.2 A hand-crafted toy example: can attention find a near-duplicate token? ──
n_tokens, token_dim = 6, 8
tokens = rng.normal(size=(n_tokens, token_dim))
tokens[4] = tokens[0] + rng.normal(scale=0.05, size=token_dim)  # token 4 ~= token 0, the rest unrelated
tokens_t = torch.tensor(tokens, dtype=torch.float32)

_, attn_weights = scaled_dot_product_attention(tokens_t, tokens_t, tokens_t)
attn_weights = attn_weights.numpy()

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(attn_weights, cmap="viridis")
ax.set_xlabel("key token"); ax.set_ylabel("query token")
ax.set_title("Attention weights (Q=K=V=raw tokens, no training)")
plt.colorbar(im, ax=ax)
plt.show()
other_tokens = [i for i in range(n_tokens) if i != 0]
best_other = other_tokens[int(np.argmax(attn_weights[0, other_tokens]))]
print("token 0's attention weights:", np.round(attn_weights[0], 3))
print(f"token 0's weight on itself: {attn_weights[0, 0]:.3f}   on its near-duplicate (token 4): {attn_weights[0, 4]:.3f}")
print("token 0's highest-weighted OTHER token:", best_other)

🔍 **What just happened?** With no training at all — $Q$, $K$, and $V$ are just the raw token vectors themselves — token 0's attention weight is split almost entirely between itself and its near-duplicate, token 4, with the four unrelated tokens together receiving only a small remainder. This is the heart of self-attention: a dot product between two vectors is large when the vectors point in similar directions, so "attend to similar content" falls directly out of the formula, with no architecture-specific rule about *where* in the sequence that content has to be.

In [ ]:
# ── 🧪 Ground-truth check ──
assert best_other == 4, "excluding self-attention, token 0's highest-weighted token should be its near-duplicate, token 4"
print("Confirmed: attention correctly identifies the near-duplicate token purely from content.")

### 📘 Lecture notes — from single-head to multi-head attention

A single attention computation can only express one notion of "similarity" at a time. **Multi-head attention** runs several independent attention computations ("heads") in parallel, each with its own learned $Q$/$K$/$V$ projections into a smaller subspace, then concatenates their outputs and projects back to the original dimension. Different heads are free to specialise — one might learn to track local nearby patches, another to track long-range structural repeats — and the final per-element representation combines all of them. The class below implements exactly this: `n_heads` independent attention computations, each operating on a `head_dim = embed_dim / n_heads` slice.

In [ ]:
# ── 2.3 Multi-head self-attention, as an nn.Module ──
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim, n_heads):
        super().__init__()
        assert embed_dim % n_heads == 0, "embed_dim must be divisible by n_heads"
        self.n_heads = n_heads
        self.head_dim = embed_dim // n_heads
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        B, N, D = x.shape

        def split_heads(t):
            return t.view(B, N, self.n_heads, self.head_dim).transpose(1, 2)

        Q, K, V = split_heads(self.q_proj(x)), split_heads(self.k_proj(x)), split_heads(self.v_proj(x))
        out, weights = scaled_dot_product_attention(Q, K, V)
        out = out.transpose(1, 2).reshape(B, N, D)
        return self.out_proj(out), weights

toy_batch = torch.randn(2, n_tokens, 16)  # (batch, sequence length, embedding dim)
mhsa = MultiHeadSelfAttention(embed_dim=16, n_heads=4)
toy_out, toy_weights = mhsa(toy_batch)
print("input shape:", tuple(toy_batch.shape), " -> output shape:", tuple(toy_out.shape))
print("attention weights shape (batch, heads, query, key):", tuple(toy_weights.shape))

🔍 **What just happened?** The output shape matches the input shape exactly — self-attention is a sequence-to-sequence operation that re-mixes information between elements without changing how many elements there are or their dimensionality, which is exactly what lets us stack many such layers (Section 3's transformer blocks) on top of each other.

## 3 · From pixels to patches: a minimal Vision Transformer (ViT)

A Vision Transformer applies self-attention to images by first **patchifying**: cutting the image into a grid of fixed-size patches, flattening each patch, and linearly projecting it into an embedding vector — turning an image into exactly the kind of "sequence of vectors" Section 2's attention mechanism expects. Three extra ingredients make this work for classification:

- a learnable **positional embedding** added to each patch embedding, since attention itself has no notion of patch order or position — without it, shuffling the patches would not change the output at all;
- a learnable **CLS token**, prepended to the sequence, whose job is to attend to every patch and accumulate a single whole-image summary (inspired directly by how BERT uses a special token for sequence classification);
- one or more stacked **transformer blocks**, each combining multi-head self-attention with a small per-element feed-forward network, both wrapped in residual connections and layer normalisation.

We will train this architecture on a task specifically chosen to need *global* reasoning: given one of our simulated nuclei images, does it contain more cells in the left half, or the right half? No single small patch can answer this on its own — it requires combining evidence from across the whole image, exactly the kind of problem self-attention's direct, single-layer long-range connections are suited to.

In [ ]:
# ── 3.1 A transformer block, and the full ViT classifier ──
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, n_heads, mlp_ratio=2):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadSelfAttention(embed_dim, n_heads)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * mlp_ratio), nn.GELU(),
            nn.Linear(embed_dim * mlp_ratio, embed_dim),
        )

    def forward(self, x):
        attn_out, weights = self.attn(self.norm1(x))
        x = x + attn_out               # residual connection around attention
        x = x + self.mlp(self.norm2(x))  # residual connection around the feed-forward network
        return x, weights

class TinyViT(nn.Module):
    def __init__(self, img_size=48, patch_size=8, embed_dim=32, n_heads=4, n_layers=2, n_classes=2):
        super().__init__()
        self.patch_size = patch_size
        n_patches = (img_size // patch_size) ** 2
        self.patch_embed = nn.Linear(patch_size * patch_size, embed_dim)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, n_patches + 1, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.blocks = nn.ModuleList([TransformerBlock(embed_dim, n_heads) for _ in range(n_layers)])
        self.head = nn.Linear(embed_dim, n_classes)

    def patchify(self, img):
        b, h, w = img.shape
        p = self.patch_size
        patches = img.unfold(1, p, p).unfold(2, p, p)  # (b, h/p, w/p, p, p)
        return patches.reshape(b, -1, p * p)

    def forward(self, img):
        x = self.patch_embed(self.patchify(img))
        b = x.shape[0]
        cls = self.cls_token.expand(b, -1, -1)
        x = torch.cat([cls, x], dim=1) + self.pos_embed
        for block in self.blocks:
            x, weights = block(x)
        return self.head(x[:, 0])  # classify from the CLS token's final representation

In [ ]:
# ── 3.2 A left/right cell-count task: simulate, train, and evaluate on held-out images ──
def make_leftright_sample(seed, size=48, n_cells=11):
    sim = simulate_nuclei_image(n_cells=n_cells, size=size, noise_sigma=0.03, seed=seed)
    cols = sim["centroids"][:, 1]
    label = 1 if (cols >= size / 2).sum() > (cols < size / 2).sum() else 0  # 1 = more cells on the right
    return sim["image"].astype(np.float32), label

train_samples = [make_leftright_sample(seed) for seed in range(160)]
test_samples = [make_leftright_sample(seed) for seed in range(900, 980)]
X_train = torch.tensor(np.stack([s[0] for s in train_samples]))
Y_train = torch.tensor([s[1] for s in train_samples])
X_test = torch.tensor(np.stack([s[0] for s in test_samples]))
Y_test = torch.tensor([s[1] for s in test_samples])
print(f"train label balance (fraction right-heavy): {Y_train.float().mean():.2f}   "
      f"test: {Y_test.float().mean():.2f}")

torch.manual_seed(SEED)
vit = TinyViT()
optimizer = torch.optim.Adam(vit.parameters(), lr=2e-3)
loss_fn = nn.CrossEntropyLoss()
for epoch in range(120):
    optimizer.zero_grad()
    logits = vit(X_train)
    loss = loss_fn(logits, Y_train)
    loss.backward()
    optimizer.step()

with torch.no_grad():
    train_acc = (vit(X_train).argmax(-1) == Y_train).float().mean().item()
    test_acc = (vit(X_test).argmax(-1) == Y_test).float().mean().item()
print(f"final training loss: {loss.item():.4f}")
print(f"training accuracy:   {train_acc:.2f}")
print(f"held-out test accuracy (seeds never trained on): {test_acc:.2f}  (chance level = 0.50)")

🧪 **Ground-truth check.** `test_samples` use seeds 900-979, which never appear among the 160 training seeds (0-159) — a genuine held-out evaluation. The task's label was derived purely from each image's ground-truth cell centroids, with no shortcut available from any single small patch in isolation.

🔍 **What just happened?** Expect training accuracy to reach (or nearly reach) 1.0 — with only 160 training images and a model this small, memorising the training set is easy — while held-out test accuracy lands clearly above the 0.50 chance level, but well short of perfect. That gap is honest and expected: Vision Transformers are well known for needing substantially more data than convolutional networks to generalise well, precisely because they start with less built-in assumption about images (no local-neighbourhood bias baked into the architecture the way a convolution kernel has one) — what they gain in flexibility, they pay for in data efficiency. Module 28 picks up exactly this thread: how large-scale and self-supervised pretraining lets transformer-based models generalise despite this.

✏️ **Your turn.** Increase `n_layers` in `TinyViT()` from 2 to 3, or `len(train_samples)` from 160 to 320 (more seeds), and retrain. Which change improves held-out accuracy more — a deeper model, or more training data? What does that tell you about where this particular model's bottleneck currently is?

## 4 · Generative modelling by denoising: the diffusion process

Module 21's VAEs and GANs both generate an image in a single forward pass from a latent code. **Diffusion models** take a different approach entirely, built from two processes:

- a **forward (noising) process**, which is not learned at all: starting from a real data point $x_0$, add a little Gaussian noise at each of $T$ steps, according to a fixed schedule $\beta_1, \dots, \beta_T$, until by step $T$ almost nothing of the original signal remains. This process has a convenient closed form that lets you jump straight to any step $t$ without simulating every step in between:

$$x_t = \sqrt{\bar\alpha_t}\, x_0 + \sqrt{1-\bar\alpha_t}\, \epsilon, \qquad \epsilon \sim \mathcal{N}(0, I), \qquad \bar\alpha_t = \textstyle\prod_{s=1}^{t}(1-\beta_s)$$

- a **reverse (denoising) process**, which *is* learned: train a network $\epsilon_\theta(x_t, t)$ to predict the noise $\epsilon$ that was added to produce $x_t$ from $x_0$, at every step $t$. Once trained, generating a new sample means starting from pure noise $x_T \sim \mathcal{N}(0, I)$ and repeatedly subtracting the network's predicted noise, step by step, until arriving back at something that looks like a draw from the original data distribution.

The network never has to generate a clean image in one shot — only ever to estimate "how much noise is in this slightly-less-noisy version," a much easier, more stable training target than GANs' adversarial min-max game or a VAE's single encode-decode bottleneck.

## 5 · A DDPM-style toy diffusion model, trained from scratch

To keep training fast while making the forward/reverse processes easy to inspect directly, this toy model generates simple 2-D points rather than full images — think of it as modelling, say, two populations of cells separated along two measured features, rather than raw pixels. Section 6 connects this directly back to how real image diffusion models (which replace the small network below with a full image-to-image U-Net) are trained the exact same way.

In [ ]:
# ── 5.1 A toy 2-D dataset: two well-separated clusters ──
def sample_toy_data(n, seed):
    local_rng = np.random.default_rng(seed)
    centers = np.array([[-2.0, -2.0], [2.0, 2.0]])
    idx = local_rng.integers(0, 2, size=n)
    pts = centers[idx] + local_rng.normal(scale=0.4, size=(n, 2))
    return torch.tensor(pts, dtype=torch.float32)

X0 = sample_toy_data(2000, seed=SEED)
plt.figure(figsize=(4, 4))
plt.scatter(X0[:, 0], X0[:, 1], s=4, alpha=0.4)
plt.title("Toy data: two clusters (this is what we want to learn to generate)")
plt.show()

In [ ]:
# ── 5.2 The forward noising process, visualised ──
N_STEPS = 100
betas = torch.linspace(1e-4, 0.02, N_STEPS)
alphas = 1 - betas
alpha_bars = torch.cumprod(alphas, dim=0)

def forward_noise(x0, t, noise=None):
    if noise is None:
        noise = torch.randn_like(x0)
    ab = alpha_bars[t].unsqueeze(-1)
    return torch.sqrt(ab) * x0 + torch.sqrt(1 - ab) * noise, noise

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
for ax, t in zip(axes, [0, 30, N_STEPS - 1]):
    t_batch = torch.full((X0.shape[0],), t, dtype=torch.long)
    xt, _ = forward_noise(X0, t_batch)
    ax.scatter(xt[:, 0], xt[:, 1], s=4, alpha=0.4)
    ax.set_title(f"t = {t}")
    ax.set_xlim(-5, 5); ax.set_ylim(-5, 5)
plt.tight_layout()
plt.show()

🔍 **What just happened?** By $t=0$ the data is exactly the two clean clusters; by the final step, it is indistinguishable from plain 2-D Gaussian noise — all trace of the two-cluster structure has been destroyed. The reverse process, trained next, has to learn to undo exactly this, one small step at a time.

In [ ]:
# ── 5.3 The epsilon-predicting network, and the DDPM training loop ──
class EpsilonNet(nn.Module):
    def __init__(self, dim=2, hidden=64, n_steps=N_STEPS):
        super().__init__()
        self.time_embed = nn.Embedding(n_steps, hidden)
        self.net = nn.Sequential(
            nn.Linear(dim + hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, dim),
        )

    def forward(self, x, t):
        return self.net(torch.cat([x, self.time_embed(t)], dim=-1))

torch.manual_seed(SEED)
eps_model = EpsilonNet()
optimizer = torch.optim.Adam(eps_model.parameters(), lr=1e-3)
batch_size = 256
for step in range(2000):
    idx = torch.randint(0, X0.shape[0], (batch_size,))
    x0 = X0[idx]
    t = torch.randint(0, N_STEPS, (batch_size,))
    xt, true_noise = forward_noise(x0, t)
    predicted_noise = eps_model(xt, t)
    loss = ((predicted_noise - true_noise) ** 2).mean()
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
print(f"final training loss (should be well below 1.0, the loss of a model that ignores xt entirely): {loss.item():.4f}")

In [ ]:
# ── 5.4 Generate new samples: start from pure noise, reverse the process step by step ──
@torch.no_grad()
def generate(n_samples):
    x = torch.randn(n_samples, 2)
    for t in reversed(range(N_STEPS)):
        t_batch = torch.full((n_samples,), t, dtype=torch.long)
        predicted_noise = eps_model(x, t_batch)
        alpha_t, alpha_bar_t, beta_t = alphas[t], alpha_bars[t], betas[t]
        coef = (1 - alpha_t) / torch.sqrt(1 - alpha_bar_t)
        mean = (x - coef * predicted_noise) / torch.sqrt(alpha_t)
        x = mean + torch.sqrt(beta_t) * torch.randn_like(x) if t > 0 else mean
    return x

generated = generate(500)
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].scatter(X0[:, 0], X0[:, 1], s=4, alpha=0.4)
axes[0].set_title("real data")
axes[1].scatter(generated[:, 0], generated[:, 1], s=4, alpha=0.4, color="crimson")
axes[1].set_title("generated (starting from pure noise)")
for ax in axes:
    ax.set_xlim(-5, 5); ax.set_ylim(-5, 5)
plt.tight_layout()
plt.show()

dist_to_pos = (generated - torch.tensor([2.0, 2.0])).norm(dim=-1)
dist_to_neg = (generated - torch.tensor([-2.0, -2.0])).norm(dim=-1)
near_a_cluster = ((dist_to_pos < 1.5) | (dist_to_neg < 1.5)).float().mean().item()
print(f"fraction of generated samples landing near one of the two true clusters: {near_a_cluster:.2f}")

🧪 **Ground-truth check.** `generate` starts every sample from `torch.randn` — pure, unstructured noise carrying no information whatsoever about the two-cluster structure. Any resemblance between the generated scatter and the real data is therefore purely a result of the trained reverse process, not of anything built into the starting point.

In [ ]:
# ── 🧪 Ground-truth check ──
assert near_a_cluster > 0.9, "the trained diffusion model should recover the two-cluster structure from pure noise"
print("Confirmed: starting from pure noise, the learned reverse process recovers the two-cluster data distribution.")

✏️ **Your turn.** Reduce `N_STEPS` from 100 to 10 (keeping everything else the same) and retrain. Does generation quality (the fraction landing near a true cluster) hold up, degrade gracefully, or break down sharply? What does this suggest about the trade-off between number of diffusion steps and sample quality?

## 6 · Where this is going: attention and diffusion in modern bioimage analysis

Real image diffusion models replace this notebook's small 2-D `EpsilonNet` with a full image-to-image network (commonly a U-Net, the same architecture family used for segmentation in Module 19, and for restoration in Module 25), predicting a noise image the same size as the input rather than a 2-D vector — but the forward process, the training objective, and the step-by-step reverse sampling procedure are exactly what you just built. This is also precisely the mechanism Module 25 pointed to when it mentioned diffusion models being repurposed as learned priors for restoration: a model trained to denoise its own forward process can, with small modifications, be steered to denoise a *real* degraded image instead.

Compared with Module 21's VAEs and GANs, diffusion models trade a simpler, more stable training objective (predict the noise — no adversarial game, no single-shot encode-decode bottleneck) for a much more expensive sampling process (many sequential denoising steps instead of one forward pass) — a real and active trade-off, not a strict improvement, which is why all three generative paradigms remain in active use.

Self-attention's contribution to modern bioimage analysis runs in a different direction: because a transformer makes no built-in assumption about local neighbourhoods, image size, or grid structure, transformer-based backbones (vision transformers, and attention mechanisms layered onto convolutional backbones) have become a key ingredient in generalist, promptable segmentation models — accepting a point, a box, or a text prompt alongside the image itself, and attending jointly over all of them. Section 3's lesson about needing much more data to make that flexibility pay off is exactly why large-scale and self-supervised pretraining, not small labelled datasets, is how those models are actually built — the subject of Module 28.

## 📌 Key takeaways

- Self-attention computes a learned, content-based weighting over every element of a sequence — $\text{softmax}(QK^\top/\sqrt{d_k})V$ — letting any two elements interact directly in a single layer, unlike a convolution's fixed, local rule.
- A Vision Transformer turns an image into a sequence by patchifying it, adds a learnable positional embedding (since attention alone has no notion of position) and a CLS token to summarise the whole sequence, and stacks transformer blocks (attention + feed-forward, both with residual connections) on top.
- ViTs make a weaker built-in assumption about images than CNNs do, which is more flexible but costs data efficiency — our tiny ViT clearly beat chance on held-out data but also clearly overfit a training set of only 160 images, a trade-off well documented at larger scale too.
- Diffusion models learn to reverse a fixed, non-learned forward noising process by training a network to predict the noise added at each step — a simpler and more stable training target than a GAN's adversarial game or a VAE's single-shot bottleneck, paid for with an expensive, multi-step sampling procedure.
- The same forward-noise/reverse-denoise mechanism built here on 2-D toy data, scaled up to a U-Net operating on real images, is what underlies modern image diffusion models — including their use as learned priors for restoration (Module 25) — while transformer backbones, for the different reason of flexible, prompt-driven attention over image content, underlie many of today's generalist segmentation foundation models (Module 28).

## ✅ Checkpoint

1. Why does scaled dot-product attention divide by $\sqrt{d_k}$ before the softmax?
   <details><summary>Answer</summary>Without the scaling, dot products grow in magnitude as the key dimension $d_k$ increases, pushing the softmax toward an extremely peaked (near one-hot) output whose gradients vanish almost everywhere, which makes the attention weights harder to learn from; dividing by $\sqrt{d_k}$ keeps the pre-softmax scores in a well-behaved range regardless of dimensionality.</details>
2. Why does a Vision Transformer need an explicit positional embedding, when a convolutional network does not need anything analogous?
   <details><summary>Answer</summary>Self-attention treats its input as an unordered set — permuting the patches would not change the attention computation at all — so position has to be injected explicitly as extra information added to each patch's embedding. A convolution kernel, by contrast, has position built directly into its structure: it is applied at a specific, fixed spatial offset relative to every pixel.</details>
3. In the left/right cell-counting task, why couldn't a model relying only on one small, isolated patch solve the task reliably?
   <details><summary>Answer</summary>Any single patch only shows a small part of the image; deciding whether the WHOLE image has more cells on the left or right requires combining evidence gathered from across the entire field of view, which is exactly the kind of global aggregation self-attention performs directly in one layer.</details>
4. What, precisely, does the network $\epsilon_\theta(x_t, t)$ learn to predict in a diffusion model, and why is that an easier training target than generating a clean sample directly?
   <details><summary>Answer</summary>It predicts the specific noise vector that was added to produce the (partially) noised input $x_t$ from the original clean data $x_0$ at diffusion step $t$ — a well-defined regression target with a known ground truth at every training step, rather than requiring the network to produce an entire realistic sample in one shot, which is a much harder, less well-specified target to optimise directly.</details>

## 🧠 Questions & discussion

- Section 3's ViT reached 100% training accuracy but well below 100% test accuracy on only 160 training images. Module 16's convolutional classifiers typically need less data to reach similar generalisation. In your own words, what architectural assumption is each model making (or not making) about images, and how does that explain the difference?
- Section 5's toy diffusion model generates 2-D points, not images. Sketch, in words, what would have to change in `EpsilonNet` (not in the forward/reverse process logic, which stays the same) to generate 48×48 images instead.
- Section 6 described diffusion models and GANs/VAEs as a genuine trade-off, not a strict improvement in either direction. For a microscopy application where generating a single new synthetic training image in real time matters more than sample diversity or training stability, which generative paradigm would you reach for first, and why?

## 🚀 Bonus — going further

- Add a third, more loosely separated Gaussian cluster to Section 5's toy dataset and retrain. Does the diffusion model still recover all three clusters cleanly, or does it start to blur them together?
- In Section 3, visualise the actual attention weights from the trained `TinyViT`'s first block for one test image (the `weights` returned alongside the block's output) as a heatmap over the 36 patches, for the CLS token's query row. Do the patches it attends to most strongly fall disproportionately in the half of the image with more cells?
- Section 2's toy attention example compared only 6 hand-crafted tokens. Repeat the same idea with 20 tokens, where 3 separate pairs are near-duplicates of each other, and verify the attention-weight heatmap now shows three distinct bright off-diagonal blocks.

## 📝 Lab note

```text
Module 27 — Attention, transformers, and diffusion
Data: SIMULATED nuclei images and 2-D toy point clouds, seed=19

Section 2 (toy attention): token 0's top match (expect token 4) [ ]

Section 3 (ViT left/right classifier):
  training accuracy    [ ]
  held-out test accuracy (chance = 0.50)  [ ]

Section 5 (toy DDPM):
  final training loss  [ ]
  fraction of generated samples near a true cluster  [ ]

Notes:
```

## 📚 References

1. Vaswani, A. et al. (2017). Attention is all you need. *NeurIPS*.
2. Dosovitskiy, A. et al. (2021). An image is worth 16x16 words: transformers for image recognition at scale (ViT). *ICLR*.
3. Devlin, J. et al. (2019). BERT: pre-training of deep bidirectional transformers for language understanding. *NAACL* (origin of the CLS-token classification trick used here).
4. Ho, J., Jain, A., & Abbeel, P. (2020). Denoising diffusion probabilistic models (DDPM). *NeurIPS*.
5. Sohl-Dickstein, J. et al. (2015). Deep unsupervised learning using nonequilibrium thermodynamics. *ICML* (the original diffusion-model framing).
6. Kawar, B., Elad, M., Ermon, S., & Song, J. (2022). Denoising diffusion restoration models. *NeurIPS* (diffusion models as learned priors for inverse problems, cross-referenced from Module 25).

## ➡️ Next

[**Module 28 — Foundation models for microscopy**](28_foundation_models_for_microscopy.ipynb): we pick up the self-supervised-pretraining thread this notebook's left/right ViT classifier left dangling — how training on vastly more data than any single labelled dataset lets transformer-based models generalise across imaging modalities, enabling promptable, zero-shot segmentation.